# Implémentation complète d'un réseau de neurones (Partie 1)

Dans ce notebook, on verra une première implémentation d'un réseau de neuronnes pour effectuer une passe avant, c'est à dire pour réaliser une prédiction à partir d'un réseau de neurones déjà entrainé.

Normalement, entrainer un réseau de neurones devrait être la première étape. Mais entrainer le réseau de neurones est aussi l'étape la plus complexe. Commençons doucement.

Dans ce notebook, nous implémenterons dans un premier temps la passe avant en utilisant des fonctions usuelles de numpy en utilisant un paradigme procédural. Nous verrons ensuite comment il est possible de passer de ce paradigme procédural à un paradigme orienté objet. Cela peut sembler superflu mais cela nous sera extrêment utile pour implémenter l'algorithme d'apprentissage d'un réseau de neurones tel qu'il est implémenté dans des librairies de Deep Learning tels que que PyTorch ou TensforFlow.

Le but de ce notebook n'est pas de fournir une implémentation optimisée mais fonctionnelle, mettant en lumière les différentes astuces utilisées dans l'industrie pour implémenter les équations mathématique dans une librairie industrielle.

## Configuration du notebook

On se limite au strict minimum: numpy !

In [170]:
import numpy as np

## Implémentation procédurale

Nous allons nous limiter à des réseaux de neurones dense et à quelques opérateurs quoi nous permettrons de faire de la classification ou de la régression. On implémentera un réseau à deux couches. Nous aurons donc besoin des opérations suivantes:

- opération linéaire
- activation ReLU
- activation sigmoide

Nous partirons du principe que les entrées du réseau de neurones sont $X_0 \in \mathcal{M}_{n,m_0}(\mathbb{R})$, c'est à dire que les $n$ différentes observations sont en ligne et les $m_0$ différentes caractéristiques d'entrée sont organisées en colonnes.

Une couche linéaire réalise l'opération suivante:

$$
    Z_1 = X_0 W_1 + b_1
$$

Avec $W_1 \in \mathcal{M}_{m_0,m_1}(\mathbb{R})$ qui représente les poids de l'opération linéaire et $b_1 \in \mathcal{M}_{1,m_1}(\mathbb{R})$ qui représente le biais.

Cette formulation, classique dans les documents traitants des réseaux de neurones, est abusive. Parce que que $X_0 W_1$ et $b_1$ ont des dimensions différentes: $n \times m_1$ pour $X_0 W_1$ et $1 \times m_1$ pour $b_1$. Cette formulation doit être comprise comme le fait que l'on ajouter $b_1$ à chaque ligne de $X_0 W_1$. C'est comme cela que fonctionne les implémentations de calcul matriciel, notamment numpy:

In [171]:
def linear(W, b, X):
    return X @ W + b

Testons cet opérateur avec 100 observations et 2 paramètres. Notre couche linéaire aura 8 neurones (et donc 8 sorties):

In [172]:
n = 100
m0 = 2
X0 = np.random.normal(0., 1., (n, m0))

In [173]:
m1 = 8
W1 = np.random.normal(0., 1., (m0, m1))
b1 = np.random.normal(0., 1., (1, m1))

In [174]:
Z1 = linear(W, b, X)

In [175]:
Z1.shape

(100, 8)

L'opérateur ReLU est très facile à implementer:

$$
    \operatorname{ReLU}(Z) = \operatorname{max}(0, Z)
$$

Cet opérateur conserve la taille de la matrice d'entrée:

In [176]:
def relu(Z):
    return np.where(Z >= 0., Z, 0.)

In [177]:
X1 = relu(Z1)
X1.shape

(100, 8)

L'opérateur logistique est aussi une formalité:

$$
    \sigma(Z) = \frac{1}{1 - \exp{(-Z)}}
$$

In [178]:
def logistic(Z):
    return 1. / (1. + np.exp(-Z))

In [179]:
X1 = relu(Z1)
X1.shape

(100, 8)

Pour implémenter un réseau de neurones à une couche cachée, pour un problème de régression:

$$
\begin{aligned}
    Z_1 &= X_0 W_1 + b_1 \\
    X_1 &= \operatorname{ReLU}(Z_1) \\
    Y   &= X_1 W_2 + b_2
\end{aligned}
$$

In [180]:
n = 100
m0 = 2
m1 = 8
m2 = 1

In [181]:
X0 = np.random.normal(0., 1., (n, m0))
W1 = np.random.normal(0., 1., (m0, m1))
b1 = np.random.normal(0., 1., (1, m1))
W2 = np.random.normal(0., 1., (m1, m2))
b2 = np.random.normal(0., 1., (1, m2))

In [182]:
Y = linear(W2, b2, relu(linear(W1, b1, X0)))
Y.shape

(100, 1)

Et pour un problème de classification:

$$
\begin{aligned}
    Z_1 &= X_0 W_1 + b_1 \\
    X_1 &= \operatorname{ReLU}(Z_1) \\
    Z_2 &= X_1 W_2 + b_2 \\
    Y   &= \sigma(Z_2)
\end{aligned}
$$

In [183]:
Y = logistic(linear(W2, b2, relu(linear(W1, b1, X0))))
Y.shape

(100, 1)

Une fois tous les opérateurs implémentés, le calcul de la passe avant est très simple à réaliser.

## Implémentation orientée objet

Un **objet**, au sens programmatique, peut être vu comme l'union d'un **état** et d'un ensemble d'**opérations**. Par exemple, l'opérateur linéaire a un état qui est lié aux poids et biais de la couche. Une opération que l'on veut pouvoir réaliser avec cet objet est de faire une passe avant.

En Python, on peut créer un tel objet comme suit. On créera une classe de base, nommée `Module` qui nous permet de définir quelques comportements par défaut:

In [184]:
class Module:
    def reset_parameters(self):
        """
        Initialisation aléatoire des paramètres. On considère que par défaut, il n'y en a pas.
        """
        pass

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées.
        
        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques
        """
        # On précise que cette méthode DOIT être définie pour chacun des modules, il n'y a pas de comportement par défaut
        raise NotImplementedError

    def __call__(self, X):
        """
        Raccourci pour réaliser une passe avant.
        """
        return self.forward(X)

L'opérateur linéaire implémente un module concret de notre réseau de neuronnes mais qui hérite de certains comportements par défaut du `Module` de base:

In [185]:
class Linear(Module):
    def __init__(self, in_features, out_features):
        """
        Constructeur.

        Ceci est la fonction qui sera appelée lorsqu'un objet est crée. self est un argument muet qui permet d'utiliser l'objet depuis 
        les différentes méthodes que l'on implémentera.

        - in_features: nombre de caractéristiques d'entrées pour la couche
        - out_features: nombre de caractéristiques de sortie pour la couche (c'est aussi le nombre de neurones)
        """
        # Cela permet de sauvegarder les arguments sur les nombres de caractéristiques directement dans l'objet, dans des attributs
        self.in_features = in_features
        self.out_features = out_features
        # Les paramètres (poids et biais) ne sont pas initialisés par défaut, leur valeur est indéfinie
        self.weight = None
        self.bias = None
        # On force une initialisation des paramètres
        self.reset_parameters()
        # Gestion de l'héritage
        super().__init__()

    def reset_parameters(self):
        """
        Initialisation aléatoire des paramètres (poids et biais).
        """
        self.weight = np.random.normal(0., 1., (self.in_features, self.out_features))
        self.bias = np.random.normal(0., 1., (1, self.out_features))

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées
        
        - X: entrées, de dimensions (n, in_features) avec n le nombre d'observations

        Retourne un tenseur de dimensions (n, out_features)
        """
        return X @ self.weight + self.bias

Pour ReLU et la fonction logistique c'est plus simple vu qu'il n'y a pas d'état à gérer:

In [186]:
class ReLU(Module):
    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées

        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques

        Retourne un tenseur de dimensions (n, out_features)
        """
        return np.where(X >= 0., X, 0.)


class Logistic(Module):
    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées

        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques

        Retourne un tenseur de dimensions (n, out_features)
        """
        return 1 / (1 + np.exp(-X))

Enfin, pour définir notre réseau de neurone complet, on peut définir une séquence qui se contentera, dans la passe avant, d'appeler tous les modules séquentiellement:

In [187]:
class Sequential(Module):
    def __init__(self, modules):
        """
        Constructeur.

        - modules: liste des modules à exécuter en séquence
        """
        self.modules = modules
        super().__init__()

    def reset_parameters(self):
        """
        Initialisation de tous les paramètres des modules de la séquence.
        """
        for module in self.modules:
            module.reset_parameters()

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées.
        
        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques
        """
        for module in self.modules:
            X = module.forward(X)
        return X

Maintenant que nous avons définit tous nos opérateurs, nous pouvons créer le même modèle que précédemment pour la régression:

In [188]:
linear1 = Linear(2, 8)
linear2 = Linear(8, 1)
relu = ReLU()
model = Sequential([linear1, relu, linear2])

La passe avant est simplement:

In [189]:
n = 100
m0 = 2
X0 = np.random.normal(0., 1., (n, m0))

In [190]:
Y = model(X0)
Y.shape

(100, 1)

Et pour la classification:

In [191]:
logistic = Logistic()
model = Sequential([linear1, relu, linear2, logistic])

Et la passe avant:

In [192]:
Y = model(X0)
Y.shape

(100, 1)

## Encapsulation supplémentaire pour les tenseurs

On verra dans la prochaine étape pour l'implémentation de l'algorithme d'apprentissage, nous aurons besoin d'un obet particulier pour les tenseurs. Nous commencerons dès maintenant avec ce formalisme, même s'il semble superflu ici !

Il est possible, en programmation orientée objet de définir le comportement des opérateurs arithmétiques pour ces objets !

In [193]:
class Tensor:
    def __init__(self, array):
        """
        Constructeur.

        - array: valeur du tenseur (array numpy)
        """
        self.data = array

    @property
    def shape(self):
        return self.data.shape
    
    def __add__(self, other):
        """
        Opération d'addition avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        if hasattr(other, "data"):
            return Tensor(self.data + other.data)
        else:
            return Tensor(self.data + other)

    def __radd__(self, other):
        """
        Opération d'addition avec un autre tenseur ou une constante.

        - other: autre tenseur ou constante
        """
        return self.__add__(other)
    
    def __matmul__(self, other):
        """
        Opération de multiplication matricielle.

        - other: autre tenseur
        """
        return Tensor(self.data @ other.data)

    def __str__(self):
        """
        Retourne une représentation textuelle du tenseur.
        """
        return str(self.data)

    def __repr__(self):
        """
        Retourne une représentation textuelle du tenseur.
        """
        return repr(self.data)

Voici concrètement ce que cela permet de faire:

In [194]:
tensor1 = Tensor(np.random.normal(0., 1., (4, 2)))
tensor1

array([[ 0.70307512, -0.50018565],
       [ 0.38705596, -1.5369233 ],
       [-2.01846143,  0.75253361],
       [-1.93419174, -1.1385615 ]])

In [195]:
tensor1 + 1

array([[ 1.70307512,  0.49981435],
       [ 1.38705596, -0.5369233 ],
       [-1.01846143,  1.75253361],
       [-0.93419174, -0.1385615 ]])

In [196]:
1 + tensor1

array([[ 1.70307512,  0.49981435],
       [ 1.38705596, -0.5369233 ],
       [-1.01846143,  1.75253361],
       [-0.93419174, -0.1385615 ]])

In [197]:
tensor2 = Tensor(np.random.normal(0., 1., (2, 3)))

In [198]:
tensor1 @ tensor2

array([[-1.52355534, -0.55071507, -0.06886499],
       [-1.00348388, -1.40190876,  0.32854327],
       [ 4.2847347 ,  0.98581298,  0.39623179],
       [ 3.86300748, -0.67498971,  0.91988286]])

Il nous faut revisiter les opérateurs précédemments définis:

In [202]:
class Linear(Module):
    def __init__(self, in_features, out_features):
        """
        Constructeur.

        Ceci est la fonction qui sera appelée lorsqu'un objet est crée. self est un argument muet qui permet d'utiliser l'objet depuis 
        les différentes méthodes que l'on implémentera.

        - in_features: nombre de caractéristiques d'entrées pour la couche
        - out_features: nombre de caractéristiques de sortie pour la couche (c'est aussi le nombre de neurones)
        """
        # Cela permet de sauvegarder les arguments sur les nombres de caractéristiques directement dans l'objet, dans des attributs
        self.in_features = in_features
        self.out_features = out_features
        # Les paramètres (poids et biais) ne sont pas initialisés par défaut, leur valeur est indéfinie
        self.weight = None
        self.bias = None
        # On force une initialisation des paramètres
        self.reset_parameters()
        # Gestion de l'héritage
        super().__init__()

    def reset_parameters(self):
        """
        Initialisation aléatoire des paramètres (poids et biais).
        """
        self.weight = Tensor(np.random.normal(0., 1., (self.in_features, self.out_features)))
        self.bias = Tensor(np.random.normal(0., 1., (1, self.out_features)))

    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées
        
        - X: entrées, de dimensions (n, in_features) avec n le nombre d'observations

        Retourne un tenseur de dimensions (n, out_features)
        """
        return X @ self.weight + self.bias

In [203]:
class ReLU(Module):
    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées

        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques

        Retourne un tenseur de dimensions (n, out_features)
        """
        return Tensor(np.where(X.data >= 0., X.data, 0.))


class Logistic(Module):
    def forward(self, X):
        """
        Implémentation de la passe avant à partir des entrées

        - X: entrées, de dimensions (n, m) avec n le nombre d'observations et m le nombre de caractéristiques

        Retourne un tenseur de dimensions (n, out_features)
        """
        return Tensor(1 / (1 + np.exp(-X.data)))

Et notre réseau de neurones pour la régression devient:

In [204]:
linear1 = Linear(2, 8)
linear2 = Linear(8, 1)
relu = ReLU()
model = Sequential([linear1, relu, linear2])

Avec la passe avant:

In [205]:
n = 100
m0 = 2
X0 = Tensor(np.random.normal(0., 1., (n, m0)))

In [206]:
Y = model(X0)
Y.shape

(100, 1)

Et pour la classification:

In [207]:
linear1 = Linear(2, 8)
linear2 = Linear(8, 1)
relu = ReLU()
logistic = Logistic()
model = Sequential([linear1, relu, linear2, logistic])

In [208]:
n = 100
m0 = 2
X0 = Tensor(np.random.normal(0., 1., (n, m0)))

In [209]:
Y = model(X0)
Y.shape

(100, 1)

## Conclusions

L'approche orienté objet permet une construction facilité et élégante du réseau de neurones. Cette facilité de définir un réseau de neurones en quelques lignes nécessite un travail de fond pour organiser proprement les opérateurs (linéaire, relu, logistique). Mais cette approche orientée objet prendra un tout autre intérêt pour implémenter l'algorithme d'apprentissage !